# init

> Router initialization for the structure decomposition workflow

In [ ]:
#| default_exp routes.init

In [ ]:
#| export
from typing import List

from fasthtml.common import APIRouter

# Import subpackage router assemblies
from cjm_fasthtml_workflow_transcript_decomp.routes.core.init import init_core_routers
from cjm_transcript_source_select.routes.init import init_selection_routers
from cjm_transcript_segmentation.routes.init import init_segmentation_routers
from cjm_transcript_vad_align.routes.init import init_alignment_routers
from cjm_transcript_review.routes.init import init_review_routers
from cjm_transcript_verify.routes.init import init_verify_routers

# Import from cjm-transcript-segment-align (extracted combined library)
from cjm_transcript_segment_align.components.handlers import (
    create_seg_mutation_wrappers,
    create_seg_init_chrome_wrapper, create_align_init_chrome_wrapper,
)
from cjm_transcript_segment_align.routes.chrome import init_chrome_router
from cjm_transcript_segment_align.routes.forced_alignment import init_forced_alignment_routers
from cjm_transcript_segment_align.services.forced_alignment import ForcedAlignmentService

from cjm_fasthtml_workflow_transcript_decomp.workflow.workflow import StructureDecompWorkflow

## Router Assembly

The `init_routers` function calls all focused router initializers and wires up cross-router dependencies.

In [ ]:
#| export
def init_routers(
    workflow: "StructureDecompWorkflow",  # The workflow instance
) -> List[APIRouter]:  # List of configured routers
    """Initialize and return all workflow routers."""
    base_prefix = workflow.config.route_prefix

    # Initialize focused routers (status, sources, audio)
    core_routers, core_routes = init_core_routers(
        workflow, f"{base_prefix}/core"
    )
    
    # Selection routers
    selection_routers, selection_urls, selection_routes, render_local_files_panel, sb_state = init_selection_routers(
        state_store=workflow.state_store,
        source_service=workflow.source_service,
        workflow_id=workflow.config.workflow_id,
        prefix=f"{base_prefix}/selection",
    )

    # Alignment routers (need align_urls for seg init KB system)
    wrapped_align_init = create_align_init_chrome_wrapper()
    
    align_routers, align_urls, align_routes = init_alignment_routers(
        state_store=workflow.state_store,
        workflow_id=workflow.config.workflow_id,
        source_service=workflow.source_service,
        alignment_service=workflow.alignment_service,
        prefix=f"{base_prefix}/align",
        audio_src_url=core_routes["audio_src"].to(),
        wrapped_init=wrapped_align_init,
    )

    # Segmentation routers (without mutation wrappers — need FA URLs first)
    seg_routers, seg_urls, seg_routes = init_segmentation_routers(
        state_store=workflow.state_store,
        workflow_id=workflow.config.workflow_id,
        source_service=workflow.source_service,
        segmentation_service=workflow.segmentation_service,
        prefix=f"{base_prefix}/seg",
        max_history_depth=workflow.config.max_history_depth,
    )

    # Forced alignment service + routes (need seg_urls)
    fa_service = ForcedAlignmentService(
        workflow.plugin_manager,
        workflow.config.fa_plugin_name,
    )
    fa_service.ensure_loaded()
    fa_is_available = fa_service.is_available()

    fa_routers = []
    fa_trigger_url = ""
    fa_toggle_url = ""

    if fa_is_available:
        fa_router, fa_routes = init_forced_alignment_routers(
            state_store=workflow.state_store,
            workflow_id=workflow.config.workflow_id,
            fa_service=fa_service,
            source_service=workflow.source_service,
            seg_urls=seg_urls,
            prefix=f"{base_prefix}/fa",
        )
        fa_routers = [fa_router]
        fa_trigger_url = fa_routes["trigger"].to()
        fa_toggle_url = fa_routes["toggle"].to()

    # Chrome switching router (needs FA URLs for toolbar extra_actions)
    chrome_router, chrome_routes = init_chrome_router(
        state_store=workflow.state_store,
        workflow_id=workflow.config.workflow_id,
        seg_urls=seg_urls,
        align_urls=align_urls,
        prefix=f"{base_prefix}/core/chrome",
        fa_trigger_url=fa_trigger_url,
        fa_toggle_url=fa_toggle_url,
        fa_available=fa_is_available,
    )
    switch_chrome_url = chrome_routes["switch_chrome"].to()

    # Create mutation wrappers (now that FA URLs are known)
    wrapped_mutations = create_seg_mutation_wrappers(
        fa_trigger_url=fa_trigger_url,
        fa_toggle_url=fa_toggle_url,
        fa_available=fa_is_available,
    )

    # Override mutation routes with wrapped versions
    seg_mutation_router = APIRouter(prefix=f"{base_prefix}/seg/workflow")

    @seg_mutation_router
    async def split(request, sess, segment_index: int):
        return await wrapped_mutations["split"](
            workflow.state_store, workflow.config.workflow_id, request, sess, segment_index,
            urls=seg_urls, max_history_depth=workflow.config.max_history_depth,
        )

    @seg_mutation_router
    async def merge(request, sess, segment_index: int):
        return await wrapped_mutations["merge"](
            workflow.state_store, workflow.config.workflow_id, request, sess, segment_index,
            urls=seg_urls, max_history_depth=workflow.config.max_history_depth,
        )

    @seg_mutation_router
    async def undo(request, sess):
        return await wrapped_mutations["undo"](
            workflow.state_store, workflow.config.workflow_id, request, sess, urls=seg_urls,
        )

    @seg_mutation_router
    async def reset(request, sess):
        return await wrapped_mutations["reset"](
            workflow.state_store, workflow.config.workflow_id, request, sess,
            urls=seg_urls, max_history_depth=workflow.config.max_history_depth,
        )

    @seg_mutation_router
    async def ai_split(request, sess):
        return await wrapped_mutations["ai_split"](
            workflow.state_store, workflow.config.workflow_id,
            workflow.segmentation_service, request, sess,
            urls=seg_urls, max_history_depth=workflow.config.max_history_depth,
        )

    # Seg init wrapper (builds combined KB system + shared chrome)
    wrapped_seg_init = create_seg_init_chrome_wrapper(
        align_urls=align_urls,
        switch_chrome_url=switch_chrome_url,
        fa_trigger_url=fa_trigger_url,
        fa_toggle_url=fa_toggle_url,
        fa_available=fa_is_available,
    )

    seg_init_router = APIRouter(prefix=f"{base_prefix}/seg/workflow")

    @seg_init_router
    async def init(request, sess):
        return await wrapped_seg_init(
            workflow.state_store, workflow.config.workflow_id,
            workflow.source_service, workflow.segmentation_service,
            request, sess, urls=seg_urls,
        )

    # Review routers
    review_routers, review_urls, review_routes = init_review_routers(
        state_store=workflow.state_store,
        workflow_id=workflow.config.workflow_id,
        prefix=f"{base_prefix}/review",
        audio_src_url=core_routes["audio_src"].to(),
        graph_service=workflow.graph_service,
        alert_container_id="commit-alert-container",
    )
    
    # Verify routers
    verify_routers, verify_urls, verify_routes = init_verify_routers(
        state_store=workflow.state_store,
        workflow_id=workflow.config.workflow_id,
        prefix=f"{base_prefix}/verify",
        verify_service=workflow.verify_service,
    )

    # Store URL bundles on workflow for renderer access
    workflow._selection_urls = selection_urls
    workflow._seg_urls = seg_urls
    workflow._align_urls = align_urls
    workflow._review_urls = review_urls
    workflow._verify_urls = verify_urls
    workflow._switch_chrome_url = switch_chrome_url
    workflow._fa_trigger_url = fa_trigger_url
    workflow._fa_toggle_url = fa_toggle_url
    workflow._fa_available = fa_is_available

    # Store selection-specific objects for renderer access
    workflow._render_local_files_panel = render_local_files_panel
    workflow._sb_state = sb_state

    # Store route dicts on workflow
    workflow._core_routes = core_routes
    workflow._selection_routes = selection_routes
    workflow._segmentation_routes = seg_routes
    workflow._alignment_routes = align_routes
    workflow._review_routes = review_routes
    workflow._verify_routes = verify_routes

    return (
        core_routers + [chrome_router] + fa_routers +
        [seg_mutation_router, seg_init_router] +
        selection_routers + seg_routers + align_routers +
        review_routers + verify_routers
    )

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()